### Student Name: Hilton Sarius
### Course: MAI5301 - Foundations Of Large Language Models
### Activity: Assigment #3

In [14]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List, Literal

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [15]:
import torch
import torch.nn as nn


class CustomLayerNorm(nn.Module):
    """
    Manual implementation of Layer Normalization.
    """
    def __init__(
        self,
        feature_dim: int,
        epsilon: float = 1e-5,
        affine: bool = True
    ):
        super().__init__()

        self.feature_dim = feature_dim
        self.epsilon = epsilon
        self.affine = affine

        self._init_affine_params()

    def _init_affine_params(self) -> None:
        if not self.affine:
            self.register_parameter("scale", None)
            self.register_parameter("shift", None)
            return

        self.scale = nn.Parameter(torch.ones(self.feature_dim))
        self.shift = nn.Parameter(torch.zeros(self.feature_dim))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # inputs: (..., C)
        stats_dim = -1

        mean = torch.mean(inputs, dim=stats_dim, keepdim=True)
        variance = torch.var(
            inputs,
            dim=stats_dim,
            keepdim=True,
            unbiased=False
        )

        normalized = (inputs - mean) / torch.sqrt(variance + self.epsilon)

        if self.affine:
            normalized = normalized * self.scale + self.shift

        return normalized


def validate_layernorm(device: str = "cpu") -> None:
    torch.manual_seed(0)

    batch_size, seq_len, channels = 2, 4, 8
    data = torch.randn(batch_size, seq_len, channels, device=device)

    custom_ln = CustomLayerNorm(channels).to(device)
    torch_ln = nn.LayerNorm(channels).to(device)

    # Synchronize affine parameters
    with torch.no_grad():
        torch_ln.weight.copy_(custom_ln.scale)
        torch_ln.bias.copy_(custom_ln.shift)

    out_custom = custom_ln(data)
    out_torch = torch_ln(data)

    difference = (out_custom - out_torch).abs().max().item()
    print(f"[LayerNorm validation] max_abs_diff = {difference:.8f} (expected ≈ 0)")



In [16]:
def fast_gelu(inputs: torch.Tensor) -> torch.Tensor:
    """
    Tanh-based GELU approximation commonly used in GPT-like architectures.
    Formula:
        0.5 * x * (1 + tanh( sqrt(2/pi) * (x + 0.044715 * x^3) ))
    """
    coeff = math.sqrt(2.0 / math.pi)
    cubic_term = inputs * inputs * inputs
    inner = coeff * (inputs + 0.044715 * cubic_term)
    return 0.5 * inputs * (1.0 + torch.tanh(inner))


class GELUActivation(nn.Module):
    """
    Stateless GELU activation wrapper.
    """
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return fast_gelu(inputs)


def validate_gelu(device: str = "cpu") -> None:
    torch.manual_seed(0)

    sample = torch.randn(1024, device=device)

    approx_out = fast_gelu(sample)
    torch_out = F.gelu(sample)  # PyTorch reference (more exact)

    diff = approx_out - torch_out
    max_diff = diff.abs().max().item()
    mean_diff = diff.abs().mean().item()

    print(
        f"[GELU validation] "
        f"max_abs_diff={max_diff:.8f}, "
        f"mean_abs_diff={mean_diff:.8f}"
    )

    # Informal comparison against ReLU
    relu_out = F.relu(sample)
    print(
        f"[ReLU vs GELU] "
        f"mean(ReLU)={relu_out.mean().item():.6f}, "
        f"mean(GELU)={torch_out.mean().item():.6f}"
    )


In [17]:
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size: int
    max_seq_len: int
    hidden_dim: int
    num_heads: int
    num_layers: int
    dropout_prob: float = 0.1
    use_bias: bool = True  # whether Linear layers include bias terms

    @classmethod
    def tiny(
        cls,
        vocab_size: int,
        max_seq_len: int,
        dropout: float = 0.1
    ) -> "ModelConfig":
        """
        Small GPT-style configuration.
        """
        return cls(
            vocab_size=vocab_size,
            max_seq_len=max_seq_len,
            hidden_dim=384,
            num_heads=6,
            num_layers=6,
            dropout_prob=dropout,
            use_bias=True,
        )

    @classmethod
    def standard(
        cls,
        vocab_size: int,
        max_seq_len: int,
        dropout: float = 0.1
    ) -> "ModelConfig":
        """
        Medium GPT-style configuration.
        """
        return cls(
            vocab_size=vocab_size,
            max_seq_len=max_seq_len,
            hidden_dim=768,
            num_heads=12,
            num_layers=12,
            dropout_prob=dropout,
            use_bias=True,
        )

In [18]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Optional, Tuple


class CausalSelfAttention(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()

        assert (
            config.hidden_dim % config.num_heads == 0
        ), "hidden_dim must be divisible by num_heads"

        self.config = config
        self.num_heads = config.num_heads
        self.head_size = config.hidden_dim // config.num_heads

        self.proj_qkv = nn.Linear(
            config.hidden_dim,
            3 * config.hidden_dim,
            bias=config.use_bias,
        )
        self.proj_out = nn.Linear(
            config.hidden_dim,
            config.hidden_dim,
            bias=config.use_bias,
        )

        self.attn_drop = nn.Dropout(config.dropout_prob)
        self.resid_drop = nn.Dropout(config.dropout_prob)

        # Precompute causal mask (non-trainable)
        causal = torch.triu(
            torch.ones(config.max_seq_len, config.max_seq_len),
            diagonal=1,
        ).bool()
        self.register_buffer("causal_mask", causal, persistent=True)

    def _split_heads(
        self, x: torch.Tensor, batch_size: int, seq_len: int
    ) -> torch.Tensor:
        return (
            x.view(batch_size, seq_len, self.num_heads, self.head_size)
            .transpose(1, 2)
        )  # (B, H, T, D)

    def forward(
        self,
        hidden_states: torch.Tensor,
        cache: Optional[Dict[str, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]]]:
        """
        hidden_states: (B, T, C)
        cache (optional):
            {
                "k": (B, H, T_cached, D),
                "v": (B, H, T_cached, D)
            }
        """
        batch_size, seq_len, embed_dim = hidden_states.size()

        qkv = self.proj_qkv(hidden_states)
        q, k, v = qkv.chunk(3, dim=-1)

        q = self._split_heads(q, batch_size, seq_len)
        k = self._split_heads(k, batch_size, seq_len)
        v = self._split_heads(v, batch_size, seq_len)

        if use_cache:
            if cache is None:
                cache = {"k": k, "v": v}
            else:
                cache["k"] = torch.cat((cache["k"], k), dim=2)
                cache["v"] = torch.cat((cache["v"], v), dim=2)

            k_full = cache["k"]
            v_full = cache["v"]
            total_len = k_full.size(2)
        else:
            k_full, v_full = k, v
            total_len = seq_len

        scores = torch.matmul(q, k_full.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_size)

        assert (
            total_len <= self.config.max_seq_len
        ), "KV cache exceeded maximum sequence length"

        row_offset = total_len - seq_len
        causal_slice = self.causal_mask[
            row_offset : row_offset + seq_len, :total_len
        ]

        scores = scores.masked_fill(
            causal_slice.view(1, 1, seq_len, total_len),
            float("-inf"),
        )

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)

        context = torch.matmul(attn, v_full)
        context = (
            context.transpose(1, 2)
            .contiguous()
            .view(batch_size, seq_len, embed_dim)
        )

        output = self.resid_drop(self.proj_out(context))
        return output, cache if use_cache else None


In [19]:
class PositionwiseMLP(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()

        expanded_dim = 4 * config.hidden_dim  # 4× expansion

        self.proj_in = nn.Linear(
            config.hidden_dim,
            expanded_dim,
            bias=config.use_bias,
        )
        self.activation = GELUActivation()
        self.proj_out = nn.Linear(
            expanded_dim,
            config.hidden_dim,
            bias=config.use_bias,
        )
        self.dropout = nn.Dropout(config.dropout_prob)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.proj_in(hidden_states)
        hidden_states = self.activation(hidden_states)
        hidden_states = self.proj_out(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return hidden_states

In [20]:
class TransformerLayer(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()

        self.norm_attn = CustomLayerNorm(config.hidden_dim)
        self.self_attn = CausalSelfAttention(config)

        self.norm_ffn = CustomLayerNorm(config.hidden_dim)
        self.mlp = PositionwiseMLP(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        cache: Optional[Dict[str, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]]]:

        attn_input = self.norm_attn(hidden_states)
        attn_output, cache = self.self_attn(
            attn_input,
            cache=cache,
            use_cache=use_cache,
        )

        hidden_states = hidden_states + attn_output
        hidden_states = hidden_states + self.mlp(
            self.norm_ffn(hidden_states)
        )

        return hidden_states, cache

In [21]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Optional, Tuple, List, Literal

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        # embeddings
        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.hidden_dim,
        )
        self.position_embedding = nn.Embedding(
            config.max_seq_len,
            config.hidden_dim,
        )
        self.embedding_dropout = nn.Dropout(config.dropout_prob)

        # transformer stack
        self.layers = nn.ModuleList(
            [TransformerLayer(config) for _ in range(config.num_layers)]
        )

        # final normalization + LM head
        self.final_norm = CustomLayerNorm(config.hidden_dim)
        self.lm_head = nn.Linear(
            config.hidden_dim,
            config.vocab_size,
            bias=False,
        )

        self.apply(self._initialize_weights)

    def _initialize_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        input_ids: torch.Tensor,
        layer_caches: Optional[List[Dict[str, torch.Tensor]]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[List[Dict[str, torch.Tensor]]]]:

        batch_size, seq_len = input_ids.size()
        assert (
            seq_len <= self.config.max_seq_len
        ), "Input sequence length exceeds configured maximum"

        positions = torch.arange(
            0,
            seq_len,
            device=input_ids.device,
        ).unsqueeze(0)

        hidden_states = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )
        hidden_states = self.embedding_dropout(hidden_states)

        updated_caches = [] if use_cache else None

        for layer_idx, layer in enumerate(self.layers):
            cache_i = None
            if use_cache and layer_caches is not None:
                cache_i = layer_caches[layer_idx]

            hidden_states, new_cache = layer(
                hidden_states,
                cache=cache_i,
                use_cache=use_cache,
            )

            if use_cache:
                updated_caches.append(
                    new_cache
                    if new_cache is not None
                    else {"k": None, "v": None}
                )

        hidden_states = self.final_norm(hidden_states)
        logits = self.lm_head(hidden_states)

        return logits, updated_caches

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
        strategy: Literal["greedy", "sample"] = "greedy",
        use_cache: bool = False,
    ) -> torch.Tensor:

        self.eval()
        caches: Optional[List[Dict[str, torch.Tensor]]] = None

        for _ in range(max_tokens):
            if use_cache and input_ids.size(1) > 1:
                model_input = input_ids[:, -1:]
            else:
                model_input = input_ids[:, -self.config.max_seq_len :]

            logits, caches = self.forward(
                model_input,
                layer_caches=caches,
                use_cache=use_cache,
            )

            logits = logits[:, -1, :]

            if temperature != 1.0:
                logits = logits / temperature

            if top_k is not None:
                top_vals, _ = torch.topk(logits, k=top_k, dim=-1)
                cutoff = top_vals[:, -1].unsqueeze(-1)
                logits = torch.where(
                    logits < cutoff,
                    torch.full_like(logits, float("-inf")),
                    logits,
                )

            probs = F.softmax(logits, dim=-1)

            if strategy == "greedy":
                next_token = torch.argmax(probs, dim=-1, keepdim=True)
            else:
                next_token = torch.multinomial(probs, num_samples=1)

            input_ids = torch.cat((input_ids, next_token), dim=1)

            if (
                input_ids.size(1) >= self.config.max_seq_len
                and not use_cache
            ):
                input_ids = input_ids[:, -self.config.max_seq_len :]

        return input_ids

In [22]:
def count_parameter_bytes(
    module: nn.Module,
    dtype: torch.dtype = torch.float32,
) -> int:
    """
    Estimate memory footprint of model parameters only.
    """
    element_bytes = torch.empty((), dtype=dtype).element_size()
    num_parameters = sum(param.numel() for param in module.parameters())
    return num_parameters * element_bytes


def estimate_adam_training_bytes(
    module: nn.Module,
    dtype: torch.dtype = torch.float32,
) -> int:
    """
    Estimate memory usage during training with Adam optimizer.

    Components:
      - parameters
      - gradients
      - first moment (m)
      - second moment (v)
    """
    param_bytes = count_parameter_bytes(module, dtype=dtype)
    return 4 * param_bytes

In [23]:
def run_demo() -> None:
    compute_device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", compute_device)

    # Basic correctness checks
    validate_layernorm(device=compute_device)
    validate_gelu(device=compute_device)

    # Minimal setup for demonstration
    vocab_size = 1000
    max_seq_len = 64

    # Compare small and medium configurations
    small_cfg = ModelConfig.tiny(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
        dropout=0.1,
    )
    medium_cfg = ModelConfig.standard(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
        dropout=0.1,
    )

    small_model = DecoderOnlyTransformer(small_cfg).to(compute_device)
    medium_model = DecoderOnlyTransformer(medium_cfg).to(compute_device)

    print("\n[Model sizes]")
    print("small params:", sum(p.numel() for p in small_model.parameters()))
    print("medium params:", sum(p.numel() for p in medium_model.parameters()))

    print("\n[Memory estimates - parameters only]")
    print(
        "small fp32 bytes:",
        count_parameter_bytes(small_model, torch.float32),
    )
    print(
        "medium fp32 bytes:",
        count_parameter_bytes(medium_model, torch.float32),
    )

In [24]:
if __name__ == "__main__":
    run_demo()


Device: cpu
[LayerNorm validation] max_abs_diff = 0.00000012 (expected ≈ 0)
[GELU validation] max_abs_diff=0.00047330, mean_abs_diff=0.00008743
[ReLU vs GELU] mean(ReLU)=0.420953, mean(GELU)=0.304491

[Model sizes]
small params: 11440128
medium params: 86641152

[Memory estimates - parameters only]
small fp32 bytes: 45760512
medium fp32 bytes: 346564608
